# 一: 环境搭建 + 数据加载 + 评估脚本

目标：跑完这个notebook，会得到：
- HotpotQA数据集已加载
- 理解数据字段结构
- EM/F1评估函数可用
- 能看到几条样本

In [ ]:
# 安装所需库（Colab里跑一次就够）
!pip install datasets transformers torch -q
!pip install rank_bm25 -q  # 第三个notebook用到BM25检索
print('安装完成！')

安装完成！


## Step 1: 加载HotpotQA数据集

In [ ]:
from datasets import load_dataset

# 加载 distractor setting（标准setting，10个候选段落，其中有干扰段落）
print('正在下载HotpotQA数据集，请稍候...')
dataset = load_dataset('hotpot_qa', 'distractor')

train_data = dataset['train']
val_data   = dataset['validation']

print(f'训练集大小: {len(train_data)}')
print(f'验证集大小: {len(val_data)}')
print('\n数据集字段:', train_data.column_names)

正在下载HotpotQA数据集，请稍候...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

distractor/train-00000-of-00002.parquet:   0%|          | 0.00/166M [00:00<?, ?B/s]

distractor/train-00001-of-00002.parquet:   0%|          | 0.00/166M [00:00<?, ?B/s]

distractor/validation-00000-of-00001.par(…):   0%|          | 0.00/27.5M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/90447 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/7405 [00:00<?, ? examples/s]

训练集大小: 90447
验证集大小: 7405

数据集字段: ['id', 'question', 'answer', 'type', 'level', 'supporting_facts', 'context']


## Step 2: 理解数据字段结构

HotpotQA每条样本包含以下字段：
- `question`: 问题
- `answer`: 标准答案
- `context`: 10个候选段落（标题 + 句子列表）
- `supporting_facts`: 真正支撑答案的段落和句子索引
- `type`: 问题类型（bridge / comparison）
- `level`: 难度（easy / medium / hard）

In [ ]:
import json

# 看一条样本的完整结构
sample = val_data[0]

print('【问题】')
print(sample['question'])
print()

print('【标准答案】')
print(sample['answer'])
print()

print('【问题类型 / 难度】')
print(f"type: {sample['type']} | level: {sample['level']}")
print()

print('【Context：10个候选段落（标题 + 句子列表）】')
for title, sentences in zip(sample['context']['title'], sample['context']['sentences']):
    print(f'  段落标题: {title}')
    print(f'  句子数: {len(sentences)}')
    print(f'  首句: {sentences[0][:80]}...')
    print()

print('【Supporting Facts：真正支撑答案的段落和句子】')
for title, sent_id in zip(sample['supporting_facts']['title'], sample['supporting_facts']['sent_id']):
    print(f'  段落: {title}, 句子编号: {sent_id}')

【问题】
Were Scott Derrickson and Ed Wood of the same nationality?

【标准答案】
yes

【问题类型 / 难度】
type: comparison | level: hard

【Context：10个候选段落（标题 + 句子列表）】
  段落标题: Ed Wood (film)
  句子数: 3
  首句: Ed Wood is a 1994 American biographical period comedy-drama film directed and pr...

  段落标题: Scott Derrickson
  句子数: 3
  首句: Scott Derrickson (born July 16, 1966) is an American director, screenwriter and ...

  段落标题: Woodson, Arkansas
  句子数: 5
  首句: Woodson is a census-designated place (CDP) in Pulaski County, Arkansas, in the U...

  段落标题: Tyler Bates
  句子数: 5
  首句: Tyler Bates (born June 5, 1965) is an American musician, music producer, and com...

  段落标题: Ed Wood
  句子数: 1
  首句: Edward Davis Wood Jr. (October 10, 1924 – December 10, 1978) was an American fil...

  段落标题: Deliver Us from Evil (2014 film)
  句子数: 3
  首句: Deliver Us from Evil is a 2014 American supernatural horror film directed by Sco...

  段落标题: Adam Collis
  句子数: 5
  首句: Adam Collis is an American filmmaker and actor....

  段落标题: Sini

## Step 3: 写一个辅助函数——把context格式化成文本

In [ ]:
def format_context(context, max_paragraphs=10):
    """
    把HotpotQA的context字段转成可读文本。

    Args:
        context: sample['context']，包含'title'和'sentences'两个列表
        max_paragraphs: 最多取几个段落（避免超出token限制）

    Returns:
        拼接好的字符串
    """
    parts = []
    titles = context['title'][:max_paragraphs]
    sentences_list = context['sentences'][:max_paragraphs]

    for title, sentences in zip(titles, sentences_list):
        paragraph_text = ' '.join(sentences)
        parts.append(f'[{title}]\n{paragraph_text}')

    return '\n\n'.join(parts)


def get_supporting_passages(sample):
    """
    提取该样本真正的支撑段落（golden passages）。
    在后续评估幻觉时会用到。
    """
    support_titles = set(sample['supporting_facts']['title'])
    passages = []

    for title, sentences in zip(
        sample['context']['title'],
        sample['context']['sentences']
    ):
        if title in support_titles:
            passages.append({
                'title': title,
                'text': ' '.join(sentences)
            })

    return passages


# 测试
print('格式化后的context（前2个段落）：')
print(format_context(sample['context'], max_paragraphs=2))
print()
print('支撑段落：')
for p in get_supporting_passages(sample):
    print(f"  [{p['title']}] {p['text'][:100]}...")

格式化后的context（前2个段落）：
[Ed Wood (film)]
Ed Wood is a 1994 American biographical period comedy-drama film directed and produced by Tim Burton, and starring Johnny Depp as cult filmmaker Ed Wood.  The film concerns the period in Wood's life when he made his best-known films as well as his relationship with actor Bela Lugosi, played by Martin Landau.  Sarah Jessica Parker, Patricia Arquette, Jeffrey Jones, Lisa Marie, and Bill Murray are among the supporting cast.

[Scott Derrickson]
Scott Derrickson (born July 16, 1966) is an American director, screenwriter and producer.  He lives in Los Angeles, California.  He is best known for directing horror films such as "Sinister", "The Exorcism of Emily Rose", and "Deliver Us From Evil", as well as the 2016 Marvel Cinematic Universe installment, "Doctor Strange."

支撑段落：
  [Scott Derrickson] Scott Derrickson (born July 16, 1966) is an American director, screenwriter and producer.  He lives ...
  [Ed Wood] Edward Davis Wood Jr. (October 10, 1924 – De

## Step 4: EM / F1 评估函数

这是HotpotQA官方评估指标的Python实现。
- **EM (Exact Match)**: 预测答案是否和标准答案完全一致（忽略大小写、标点）
- **F1**: 预测答案和标准答案的词重叠率

In [ ]:
import re
import string
from collections import Counter


def normalize_answer(s):
    """标准化答案：小写、去标点、去冠词、去多余空格"""
    def remove_articles(text):
        return re.sub(r'\b(a|an|the)\b', ' ', text)

    def white_space_fix(text):
        return ' '.join(text.split())

    def remove_punc(text):
        exclude = set(string.punctuation)
        return ''.join(ch for ch in text if ch not in exclude)

    def lower(text):
        return text.lower()

    return white_space_fix(remove_articles(remove_punc(lower(s))))


def get_tokens(s):
    if not s:
        return []
    return normalize_answer(s).split()


def compute_exact(prediction, ground_truth):
    """计算单条样本的EM"""
    return int(normalize_answer(prediction) == normalize_answer(ground_truth))


def compute_f1(prediction, ground_truth):
    """计算单条样本的F1"""
    pred_tokens  = get_tokens(prediction)
    truth_tokens = get_tokens(ground_truth)
    common = Counter(pred_tokens) & Counter(truth_tokens)
    num_same = sum(common.values())

    if len(pred_tokens) == 0 or len(truth_tokens) == 0:
        return int(pred_tokens == truth_tokens)
    if num_same == 0:
        return 0.0

    precision = num_same / len(pred_tokens)
    recall    = num_same / len(truth_tokens)
    f1 = (2 * precision * recall) / (precision + recall)
    return f1


def evaluate(predictions, ground_truths):
    """
    批量评估。

    Args:
        predictions: 预测答案列表
        ground_truths: 标准答案列表

    Returns:
        dict，包含平均EM和平均F1
    """
    assert len(predictions) == len(ground_truths), '长度不一致'

    em_scores = [compute_exact(p, g) for p, g in zip(predictions, ground_truths)]
    f1_scores = [compute_f1(p, g)    for p, g in zip(predictions, ground_truths)]

    return {
        'em':  sum(em_scores) / len(em_scores),
        'f1':  sum(f1_scores) / len(f1_scores),
        'n':   len(predictions)
    }


# 测试评估函数
test_preds  = ['Edward Jenner', 'the united states', 'yes']
test_truths = ['Edward Jenner', 'United States', 'yes']

results = evaluate(test_preds, test_truths)
print(f"测试评估结果：EM={results['em']:.3f}, F1={results['f1']:.3f}, N={results['n']}")
print('（EM应该是0.667，F1应该是0.889）')

测试评估结果：EM=1.000, F1=1.000, N=3
（EM应该是0.667，F1应该是0.889）


## Step 5: 抽取实验子集

验证集有7405条，全跑太慢也太贵。我们取200条做实验，但要保证各类型分布均衡。

In [ ]:
import random
random.seed(42)  # 固定随机种子，保证可复现

# 按类型分层采样：bridge题（多跳推理）和comparison题
bridge_samples    = [s for s in val_data if s['type'] == 'bridge']
comparison_samples = [s for s in val_data if s['type'] == 'comparison']

print(f'验证集总数: {len(val_data)}')
print(f'Bridge类型: {len(bridge_samples)}')
print(f'Comparison类型: {len(comparison_samples)}')

# 取160条bridge + 40条comparison，共200条
sample_bridge     = random.sample(bridge_samples,     160)
sample_comparison = random.sample(comparison_samples,  40)
eval_samples      = sample_bridge + sample_comparison

random.shuffle(eval_samples)  # 打乱顺序

print(f'\n实验子集大小: {len(eval_samples)}')
print('分布：', {
    'bridge': sum(1 for s in eval_samples if s['type'] == 'bridge'),
    'comparison': sum(1 for s in eval_samples if s['type'] == 'comparison')
})

验证集总数: 7405
Bridge类型: 5918
Comparison类型: 1487

实验子集大小: 200
分布： {'bridge': 160, 'comparison': 40}


In [ ]:
import json

# 保存子集到文件，后续notebook直接加载，不用重新采样
with open('eval_samples.json', 'w') as f:
    json.dump(eval_samples, f, ensure_ascii=False, indent=2)

print('已保存到 eval_samples.json')
print(f'共 {len(eval_samples)} 条样本，后续notebook直接load这个文件')

已保存到 eval_samples.json
共 200 条样本，后续notebook直接load这个文件


跑完这个第一部分后，确认以下几点：
1. `eval_samples.json` 已生成，200条样本
2. `evaluate()` 函数测试通过（EM=0.667, F1=0.889）
3. 你能看懂一条HotpotQA样本的结构（question / answer / context / supporting_facts）

# 二: Naive Baseline

目标：把所有候选段落直接拼接，喂给模型，得到第一个EM/F1数字。

Qwen2.5-7B-Instruct 本地加载

In [ ]:
import json
import re
import string
import time
from collections import Counter

# 加载之前保存的200条样本
with open('eval_samples.json', 'r') as f:
    eval_samples = json.load(f)

print(f'加载成功，共 {len(eval_samples)} 条样本')

加载成功，共 200 条样本


## step1：加载模型

In [ ]:
# Qwen2.5-7B-Instruct 本地加载
# 注意：需要Colab Pro或GPU实例，下载约15GB
# ============================================================
!pip install transformers accelerate -q
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = 'Qwen/Qwen2.5-7B-Instruct'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name, torch_dtype=torch.float16, device_map='auto'
)

def call_model(prompt, max_tokens=64):
    messages = [{'role': 'user', 'content': prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([text], return_tensors='pt').to(model.device)
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=max_tokens, temperature=0.01, do_sample=False)
    gen = output[0][inputs.input_ids.shape[1]:]
    return tokenizer.decode(gen, skip_special_tokens=True).strip()

print('方案B: Qwen2.5-7B 已加载')

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

方案B: Qwen2.5-7B 已加载


## 构建Naive Baseline的Prompt

Naive Baseline的逻辑非常简单：把所有10个候选段落全部拼接，直接要求模型给出答案和支撑句。
这是我们的下限——任何方法都要比这强。

In [ ]:
def build_index_hint(context, max_paragraphs=10):
    """
    把context格式化成带句子编号的文本，方便模型引用。
    例如：
    [Scott Derrickson]
      [0] Scott Derrickson is an American director...
      [1] He is best known for...
    """
    parts = []
    for title, sentences in zip(
        context['title'][:max_paragraphs],
        context['sentences'][:max_paragraphs]
    ):
        lines = [f"[{i}] {s}" for i, s in enumerate(sentences)]
        parts.append(f"[{title}]\n" + '\n'.join(lines))
    return '\n\n'.join(parts)


def parse_sp_output(raw_output, valid_titles):
    """
    从模型输出里解析支撑句预测。

    Args:
        raw_output: 模型原始输出字符串
        valid_titles: 该样本中所有合法的段落标题（用于过滤幻觉标题）

    Returns:
        answer: 解析出的答案字符串
        sp: [[title, sent_id], ...] 格式的支撑句列表
    """
    answer = ''
    sp = []

    # 解析Answer
    ans_match = re.search(r'Answer:\s*(.+?)(?:\n|$)', raw_output)
    if ans_match:
        answer = ans_match.group(1).strip()

    # 解析Supporting facts
    sp_match = re.search(r'Supporting facts:\s*(.+?)(?:\n|$)', raw_output, re.DOTALL)
    if sp_match:
        sp_text = sp_match.group(1).strip()
        # 匹配 [Title, number] 格式
        pattern = r'\[([^\]]+),\s*(\d+)\]'
        matches = re.findall(pattern, sp_text)
        for title, sent_id in matches:
            title = title.strip()
            # 只保留合法标题（防止模型幻觉出不存在的段落）
            if title in valid_titles:
                sp.append([title, int(sent_id)])

    return answer, sp

In [ ]:
def build_naive_prompt(sample):
    context_text = format_context(sample['context'], max_paragraphs=10)

    # 把段落标题和句子编号列出来，方便模型引用
    index_text = build_index_hint(sample['context'])

    prompt = f"""Answer the question based on the provided passages.

Output format (strictly follow this):
Answer: <your answer>
Supporting facts: [Title, sentence_index], [Title, sentence_index], ...

Rules:
- Answer should be as short as possible (a few words)
- Supporting facts must use exact titles from the passages
- sentence_index starts from 0

Passages (with sentence indices):
{index_text}

Question: {sample['question']}"""
    return prompt


# 预览一下prompt长什么样
sample = eval_samples[0]
prompt = build_naive_prompt(sample)
print('=== Prompt预览（截取前600字符）===')
print(prompt[:600])
print('...')
print(f'\n完整prompt长度: {len(prompt)} 字符')
print(f'标准答案: {sample["answer"]}')

=== Prompt预览（截取前600字符）===
Answer the question based on the provided passages.

Output format (strictly follow this):
Answer: <your answer>
Supporting facts: [Title, sentence_index], [Title, sentence_index], ...

Rules:
- Answer should be as short as possible (a few words)
- Supporting facts must use exact titles from the passages
- sentence_index starts from 0

Passages (with sentence indices):
[Moe Szyslak]
[0] Morris "Moe" Szyslak is a fictional character from the American animated television series "The Simpsons".
[1]  He is voiced by Hank Azaria and first appeared in the series premiere episode "Simpsons Roasting o
...

完整prompt长度: 6512 字符
标准答案: Daniel Louis Castellaneta


## 跑Naive Baseline（200条）

In [ ]:
import time

naive_predictions   = []
naive_ground_truths = []
naive_records       = []

print('开始跑Naive Baseline...')
print('（200条，预计需要5-10分钟，取决于API速度）\n')

for i, sample in enumerate(eval_samples):
    prompt       = build_naive_prompt(sample)
    valid_titles = sample['context']['title']

    try:
        raw_output     = call_model(prompt, max_tokens=128)  # 128给sp留空间
        pred, sp_pred  = parse_sp_output(raw_output, valid_titles)
    except Exception as e:
        print(f'  样本{i}出错: {e}')
        raw_output, pred, sp_pred = '', '', []

    naive_predictions.append(pred)
    naive_ground_truths.append(sample['answer'])
    naive_records.append({
        'id':           i,
        'question':     sample['question'],
        'answer':       sample['answer'],
        'prediction':   pred,
        'sp_prediction': sp_pred,
        'raw_output':   raw_output,
        'type':         sample['type'],
        'level':        sample['level'],
        'em':           compute_exact(pred, sample['answer']),
        'f1':           compute_f1(pred, sample['answer'])
    })

    if (i + 1) % 20 == 0:
        partial = evaluate(naive_predictions, naive_ground_truths)
        sp_coverage = sum(1 for r in naive_records if len(r['sp_prediction']) > 0)
        print(f'[{i+1}/200] EM={partial["em"]:.3f}, F1={partial["f1"]:.3f} | sp有输出: {sp_coverage}/{i+1}')

    time.sleep(0.3)

print('\n完成！')

# 检查sp解析情况
sp_empty = sum(1 for r in naive_records if len(r['sp_prediction']) == 0)
print(f'\nsp解析失败（空列表）的样本数: {sp_empty}/200')
print('如果这个数字很大（>50），说明模型没有按格式输出，需要调整prompt')

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


开始跑Naive Baseline...
（200条，预计需要5-10分钟，取决于API速度）

[20/200] EM=0.250, F1=0.358 | sp有输出: 19/20
[40/200] EM=0.325, F1=0.398 | sp有输出: 39/40
[60/200] EM=0.283, F1=0.338 | sp有输出: 58/60
  样本76出错: CUDA out of memory. Tried to allocate 1.97 GiB. GPU 0 has a total capacity of 14.56 GiB of which 1.48 GiB is free. Including non-PyTorch memory, this process has 13.08 GiB memory in use. Of the allocated memory 12.35 GiB is allocated by PyTorch, and 612.80 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)
[80/200] EM=0.250, F1=0.309 | sp有输出: 77/80
[100/200] EM=0.280, F1=0.327 | sp有输出: 95/100
  样本105出错: CUDA out of memory. Tried to allocate 764.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 475.81 MiB is free. Including non-PyTorch memory, this process has 14.10 Gi

## 保存成官方eval脚本要求的格式
跑完之后，把结果转成官方 eval.py 要求的格式：

In [ ]:
def save_for_official_eval(records, output_path):
    """
    生成官方eval.py要求的prediction文件格式：
    {
        "answer": {"id": "答案", ...},
        "sp":     {"id": [["标题", 句子号], ...], ...}
    }
    """
    prediction = {
        "answer": {},
        "sp": {}
    }
    for r in records:
        qid = str(r['id'])
        prediction["answer"][qid] = r['prediction']
        prediction["sp"][qid]     = r['sp_prediction']

    with open(output_path, 'w') as f:
        json.dump(prediction, f, ensure_ascii=False, indent=2)

    print(f'已保存到 {output_path}')
    print(f'格式检查：answer数={len(prediction["answer"])}, sp数={len(prediction["sp"])}')


# 调用
save_for_official_eval(naive_records, 'naive_prediction.json')
# 然后用官方脚本评估：
# python eval.py naive_prediction.json gold.json

已保存到 naive_prediction.json
格式检查：answer数=200, sp数=200


## 直接评估

In [ ]:
import re

# ============================================================
# sp评估函数（对应官方eval.py的update_sp逻辑）
# ============================================================
def compute_sp_metrics(sp_pred, sp_gold):
    """
    sp_pred: [[title, sent_id], ...]
    sp_gold: [[title, sent_id], ...]
    """
    cur_sp_pred  = set(map(tuple, sp_pred))
    gold_sp_pred = set(map(tuple, sp_gold))
    tp, fp, fn = 0, 0, 0
    for e in cur_sp_pred:
        if e in gold_sp_pred: tp += 1
        else:                  fp += 1
    for e in gold_sp_pred:
        if e not in cur_sp_pred: fn += 1
    prec   = tp / (tp + fp) if tp + fp > 0 else 0.0
    recall = tp / (tp + fn) if tp + fn > 0 else 0.0
    f1     = 2 * prec * recall / (prec + recall) if prec + recall > 0 else 0.0
    em     = 1.0 if fp + fn == 0 else 0.0
    return {'sp_em': em, 'sp_f1': f1, 'sp_prec': prec, 'sp_recall': recall}


def compute_joint_metrics(ans_f1, ans_prec, ans_recall, sp_em, sp_prec, sp_recall, ans_em):
    joint_prec   = ans_prec   * sp_prec
    joint_recall = ans_recall * sp_recall
    joint_f1     = 2 * joint_prec * joint_recall / (joint_prec + joint_recall) if joint_prec + joint_recall > 0 else 0.0
    joint_em     = ans_em * sp_em
    return {'joint_em': joint_em, 'joint_f1': joint_f1,
            'joint_prec': joint_prec, 'joint_recall': joint_recall}


# ============================================================
# 给每条record补充sp/joint指标
# ============================================================
from collections import Counter

def f1_score_full(prediction, ground_truth):
    """返回(f1, prec, recall)，对应官方eval.py"""
    def norm(s):
        s = re.sub(r'\b(a|an|the)\b', ' ', s.lower())
        s = ''.join(ch for ch in s if ch not in set(string.punctuation))
        return ' '.join(s.split())

    if norm(prediction) in ['yes','no','noanswer'] and norm(prediction) != norm(ground_truth):
        return 0.0, 0.0, 0.0
    if norm(ground_truth) in ['yes','no','noanswer'] and norm(prediction) != norm(ground_truth):
        return 0.0, 0.0, 0.0

    p_toks = norm(prediction).split()
    g_toks = norm(ground_truth).split()
    common = Counter(p_toks) & Counter(g_toks)
    num_same = sum(common.values())
    if num_same == 0:
        return 0.0, 0.0, 0.0
    prec   = num_same / len(p_toks)
    recall = num_same / len(g_toks)
    f1     = 2 * prec * recall / (prec + recall)
    return f1, prec, recall


In [ ]:
for r in naive_records:
    # 取gold sp
    idx      = r['id']
    sample   = eval_samples[idx]
    sp_gold  = list(zip(
        sample['supporting_facts']['title'],
        sample['supporting_facts']['sent_id']
    ))

    # ans完整指标
    ans_f1, ans_prec, ans_recall = f1_score_full(r['prediction'], r['answer'])
    ans_em = compute_exact(r['prediction'], r['answer'])

    # sp指标
    sp_metrics = compute_sp_metrics(r['sp_prediction'], sp_gold)

    # joint指标
    joint_metrics = compute_joint_metrics(
        ans_f1, ans_prec, ans_recall,
        sp_metrics['sp_em'], sp_metrics['sp_prec'], sp_metrics['sp_recall'],
        ans_em
    )

    # 写回record
    r.update(sp_metrics)
    r.update(joint_metrics)


In [ ]:
# ============================================================
# 汇总打印
# ============================================================
def avg(records, key):
    return sum(r[key] for r in records) / len(records)

naive_results = evaluate(naive_predictions, naive_ground_truths)

print('=' * 52)
print('Naive Baseline 最终结果')
print('=' * 52)
print(f"{'指标':<16} {'EM':>8} {'F1':>8}")
print('-' * 36)
print(f"{'ans':<16} {avg(naive_records,'em'):>8.4f} {avg(naive_records,'f1'):>8.4f}")
print(f"{'sp':<16} {avg(naive_records,'sp_em'):>8.4f} {avg(naive_records,'sp_f1'):>8.4f}")
print(f"{'joint':<16} {avg(naive_records,'joint_em'):>8.4f} {avg(naive_records,'joint_f1'):>8.4f}")
print('=' * 52)
print(f"N: {len(naive_records)}条")

# 分类型统计
print()
for qtype in ['bridge', 'comparison']:
    tr = [r for r in naive_records if r['type'] == qtype]
    print(f"{qtype} (n={len(tr)}):")
    print(f"  ans   EM={avg(tr,'em'):.3f}       F1={avg(tr,'f1'):.3f}")
    print(f"  sp    EM={avg(tr,'sp_em'):.3f}    F1={avg(tr,'sp_f1'):.3f}")
    print(f"  joint EM={avg(tr,'joint_em'):.3f} F1={avg(tr,'joint_f1'):.3f}")


Naive Baseline 最终结果
指标                     EM       F1
------------------------------------
ans                0.2300   0.2846
sp                 0.1500   0.5116
joint              0.0300   0.1479
N: 200条

bridge (n=160):
  ans   EM=0.212       F1=0.278
  sp    EM=0.075    F1=0.470
  joint EM=0.013 F1=0.139
comparison (n=40):
  ans   EM=0.300       F1=0.312
  sp    EM=0.450    F1=0.677
  joint EM=0.100 F1=0.183


In [ ]:
# 保存
with open('naive_results.json', 'w') as f:
    json.dump({'summary': {
        'ans_em':    avg(naive_records,'em'),
        'ans_f1':    avg(naive_records,'f1'),
        'sp_em':     avg(naive_records,'sp_em'),
        'sp_f1':     avg(naive_records,'sp_f1'),
        'joint_em':  avg(naive_records,'joint_em'),
        'joint_f1':  avg(naive_records,'joint_f1'),
        'n':         len(naive_records)
    }, 'records': naive_records}, f, ensure_ascii=False, indent=2)

print('\n结果已保存到 naive_results.json')

# 答对/答错例子（同时展示sp情况）
correct = [r for r in naive_records if r['em'] == 1][:3]
wrong   = [r for r in naive_records if r['em'] == 0][:3]

print('\n【答对的例子】')
for r in correct:
    print(f"  Q:  {r['question'][:65]}")
    print(f"  预测: {r['prediction']}  |  答案: {r['answer']}")
    print(f"  sp预测: {r['sp_prediction']}")
    print(f"  sp_f1={r['sp_f1']:.2f}  joint_f1={r['joint_f1']:.2f}\n")

print('\n【答错的例子】')
for r in wrong:
    print(f"  Q:  {r['question'][:65]}")
    print(f"  预测: {r['prediction']}  |  答案: {r['answer']}")
    print(f"  sp预测: {r['sp_prediction']}")
    print(f"  sp_f1={r['sp_f1']:.2f}  joint_f1={r['joint_f1']:.2f}\n")


结果已保存到 naive_results.json

【答对的例子】
  Q:  What is the name of the so-called reform opera for Vienna that ca
  预测: Alceste  |  答案: "Alceste"
  sp预测: [['Orfeo ed Euridice', 2]]
  sp_f1=0.50  joint_f1=0.50

  Q:  The organization that Nicolae Titulescu served two terms as presi
  预测: 10 January 1920  |  答案: 10 January 1920
  sp预测: [['League of Nations', 0]]
  sp_f1=0.67  joint_f1=0.67

  Q:  Which one of the Long Island Herald newspaper chain serves a vill
  预测: Nassau Herald  |  答案: The Nassau Herald
  sp预测: [['Lawrence, Nassau County, New York', 1], ['Nassau Herald', 0]]
  sp_f1=0.80  joint_f1=0.80


【答错的例子】
  Q:  Who voices Homer Simpson's character on the animated television s
  预测: Dan Castellaneta  |  答案: Daniel Louis Castellaneta
  sp预测: []
  sp_f1=0.00  joint_f1=0.00

  Q:  What country of origin does  Dana Ivey and Two Weeks Notice have 
  预测:   |  答案: American
  sp预测: [['Dana Ivey', 0], ['Two Weeks Notice', 0]]
  sp_f1=1.00  joint_f1=0.00

  Q:  In addition to the best known com

In [ ]:
# 计算最终结果
naive_results = evaluate(naive_predictions, naive_ground_truths)

print('=' * 40)
print('Naive Baseline 最终结果')
print('=' * 40)
print(f"EM:  {naive_results['em']:.4f} ({naive_results['em']*100:.1f}%)")
print(f"F1:  {naive_results['f1']:.4f} ({naive_results['f1']*100:.1f}%)")
print(f"N:   {naive_results['n']}条")

# 分类型统计
print()
for qtype in ['bridge', 'comparison']:
    type_records = [r for r in naive_records if r['type'] == qtype]
    em = sum(r['em'] for r in type_records) / len(type_records)
    f1 = sum(r['f1'] for r in type_records) / len(type_records)
    print(f"{qtype}: EM={em:.3f}, F1={f1:.3f} (n={len(type_records)})")

In [ ]:
# 保存结果，供后续对比
with open('naive_results.json', 'w') as f:
    json.dump({
        'summary': naive_results,
        'records': naive_records
    }, f, ensure_ascii=False, indent=2)

print('结果已保存到 naive_results.json')

# 看几个答对和答错的例子
correct = [r for r in naive_records if r['em'] == 1][:3]
wrong   = [r for r in naive_records if r['em'] == 0][:3]

print('\n【答对的例子】')
for r in correct:
    print(f"  Q: {r['question'][:60]}")
    print(f"  预测: {r['prediction']}  |  答案: {r['answer']}\n")

print('\n【答错的例子】')
for r in wrong:
    print(f"  Q: {r['question'][:60]}")
    print(f"  预测: {r['prediction']}  |  答案: {r['answer']}\n")